In [ ]:
import ROOT
%load_ext JupyROOT

In [ ]:
%%cpp
const float A_low = 4.;
const float A_high = 6.;

const float B_low = 5.;
const float B_high = 9;

In [ ]:
  %%cpp
  TObject *obj = gROOT->FindObject("cPart");
  if(obj)
  {
      delete obj;
  }
  gStyle->SetOptStat(0);
  TFile *file = new TFile("HEPData-ins860416-v1-Table_4.root", "READ");

  TDirectory *dir = (TDirectory*)file->Get("Table 4");

  TH1F *chPart = (TH1F*)dir->Get("Hist1D_y1;1");


  TF1 *pt_func = new TF1("pt_func", "[0]*x^(-[1])", 3., 10.);
  pt_func->SetParameters(0.9, 6.63);
  chPart->Fit(pt_func, "0WR", "", 3., 10.);
  chPart->SetTitle("Charged Particles;#it{p}_{T} (GeV/#it{c});Probability Density");
  chPart->GetXaxis()->SetRangeUser(3., 10.);
  TCanvas *cPart = new TCanvas("cPart", "cPart", 550, 550);
  cPart->cd();
  gPad->SetLogy();
  gPad->SetLeftMargin(0.15);
  chPart->SetMarkerStyle(kFullCircle);
  chPart->SetMarkerColor(kBlack);
  chPart->SetLineColor(kBlack);
  chPart->SetLineWidth(2);
  chPart->Draw();

  TF1 *f_shade = (TF1*)pt_func->Clone("f_shade");
  f_shade->SetRange(A_low, A_high);
  f_shade->SetFillStyle(1001);
  Int_t trans_blue = TColor::GetColorTransparent(kBlue, 0.4);
  f_shade->SetFillColor(trans_blue);
  f_shade->Draw("same");

  TF1 *f_shade2 = (TF1*)pt_func->Clone("f_shade2");
  f_shade2->SetRange(B_low, B_high);
  f_shade2->SetFillStyle(1001);
  Int_t trans_red = TColor::GetColorTransparent(kRed, 0.4);
  f_shade2->SetFillColor(trans_red);
  f_shade2->Draw("same");

  TF1 *f_shade_inter = (TF1*)pt_func->Clone("f_shade_inter");
  f_shade_inter->SetFillStyle(1001);
  f_shade_inter->SetRange(B_low, A_high);
  Int_t trans_violet = TColor::GetColorTransparent(kViolet, 0.4);
  f_shade_inter->SetFillColor(trans_violet);
  
  if(A_high < B_low)
  {
      f_shade_inter->SetRange(A_high, B_low);
      Int_t trans_green = TColor::GetColorTransparent(kGreen, 0.4);
      f_shade_inter->SetFillColor(trans_green);
  }
  
  
  f_shade_inter->Draw("same");
  cPart->Draw();

  TLegend *leg = new TLegend(0.3, 0.75, 0.89, 0.89);
  leg->SetFillColor(kWhite);
  leg->SetBorderSize(0);
  leg->SetTextSize(0.025);
  leg->AddEntry(chPart, "Charged particles", "lp");
  leg->AddEntry(pt_func, "Phys.Lett.B 693 (2010) 53-68, 2010", "");
  leg->Draw("same");

  float PA = pt_func->Integral(A_low, A_high);
  float PB = pt_func->Integral(B_low, B_high);
  float PAintB = pt_func->Integral(B_low, A_high);
  if(A_high < B_low)
  {
      PAintB = pt_func->Integral(A_high, B_low);
  }
  float PAuniB = PA + PB - PAintB;

  std::string A_interval = Form("P(A)[%.1lf to %.1lf GeV/c]: ", A_low, A_high);
  std::string B_interval = Form("P(B)[%.1lf to %.1lf GeV/c]: ", B_low, B_high);
  std::string A_int_B = "";
  std::string A_uni_B = "";
  if(A_high > B_low)
  {
      A_int_B = Form("P(A\u2229B)[%.1lf to %.1lf GeV/c]: ", B_low, A_high);
      A_uni_B = Form("P(A\u222AB)[%.1lf to %.1lf GeV/c]: ", A_low, B_high);
  }
  else
  {
      A_int_B = Form("P(C)[%.1lf to %.1lf GeV/c]: ", B_low, A_high);
      A_uni_B = Form("P(A\u222AB\u222AC)[%.1lf to %.1lf GeV/c]: ", A_low, B_high);
  }

  std::cout << "" << std::endl;

  std::cout << "pp collisions at sqrt(s) = 900 GeV. |η(ch particles)| < 0.8" << std::endl;
  
  std::cout << "" << std::endl;

  std::cout << A_interval << PA << std::endl;
  std::cout << B_interval << PB << std::endl;
  std::cout << A_int_B << PAintB << std::endl;
  std::cout << A_uni_B << PAuniB << std::endl;
  